# Exploratory data analysis: IBM AMLworld HI-Small

Stage 1 asks four questions of the data before any rules or models are built:

1. How rare is laundering?
2. What time period does the data really cover, and how evenly?
3. What do laundering transactions and patterns look like?
4. How many accounts would count as mules under each candidate account-level label?

The data is synthetic (IBM's AMLworld simulator), so these figures describe the simulation,
not a real bank. Data is loaded through `mule_detection.data.load`, which names columns by
position and strips zero-padding from bank IDs (see `docs/decisions.md`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mule_detection.data import load

pd.set_option("display.float_format", "{:,.4f}".format)
tx = load.load_transactions()
accounts = load.load_accounts()
tx.shape, accounts.shape

## 1. Overview: how rare is laundering?

Laundering and normal transactions are the two classes the project has to tell apart. When one
class is far rarer than the other (class imbalance), accuracy becomes misleading and
evaluation has to focus on the rare class.

In [ ]:
n_tx = len(tx)
n_laundering = int(tx["is_laundering"].sum())
pd.Series(
    {
        "transactions": n_tx,
        "laundering transactions": n_laundering,
        "laundering rate (%)": round(100 * n_laundering / n_tx, 4),
        "first timestamp": tx["timestamp"].min(),
        "last timestamp": tx["timestamp"].max(),
    }
)

## 2. Transactions time period

In [ ]:
daily = (
    tx.assign(date=tx["timestamp"].dt.date)
    .groupby("date")
    .agg(transactions=("is_laundering", "size"), laundering=("is_laundering", "sum"))
)
daily["laundering rate (%)"] = 100 * daily["laundering"] / daily["transactions"]
daily

In [ ]:
from matplotlib.ticker import FuncFormatter

def compact(value, _):
    if value >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"
    if value >= 1_000:
        return f"{value / 1_000:,.0f}k"
    return f"{value:,.0f}"

days = [d.strftime("%a %d %b") for d in daily.index]
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
panels = [
    ("transactions", "Transactions per day", "Transactions", "#4C72B0"),
    ("laundering", "Laundering transactions per day", "Laundering transactions", "#C44E52"),
]
for ax, (col, title, ylabel, colour) in zip(axes, panels):
    ax.bar(days, daily[col], color=colour)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(FuncFormatter(compact))
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
axes[1].tick_params(axis="x", rotation=45)
plt.setp(axes[1].get_xticklabels(), ha="right")
fig.tight_layout()

The bars after 10 September are too small to see on the top chart: those days hold between
11 and 396 transactions each (see the table above).

Normal activity stops after 10 September; the days after that hold only a small
number of transactions, most of them laundering. The tail is summarised below.

In [ ]:
tail = daily.loc[daily.index > pd.Timestamp("2022-09-10").date()]
pd.Series(
    {
        "days after 10 Sept": len(tail),
        "transactions": int(tail["transactions"].sum()),
        "laundering": int(tail["laundering"].sum()),
        "laundering rate (%)": round(100 * tail["laundering"].sum() / tail["transactions"].sum(), 2),
        "share of all laundering (%)": round(100 * tail["laundering"].sum() / n_laundering, 2),
    }
)

## 3. Payment formats and currencies

In [ ]:
def rate_table(col):
    t = tx.groupby(col, observed=True)["is_laundering"].agg(transactions="size", laundering="sum")
    t["share of all (%)"] = 100 * t["transactions"] / n_tx
    t["laundering rate (%)"] = 100 * t["laundering"] / t["transactions"]
    t["share of laundering (%)"] = 100 * t["laundering"] / n_laundering
    return t.sort_values("transactions", ascending=False)

rate_table("payment_format")

In [ ]:
rate_table("payment_currency")

In [ ]:
cross_currency = tx["payment_currency"].astype(str) != tx["receiving_currency"].astype(str)
pd.crosstab(cross_currency.rename("cross-currency"), tx["is_laundering"], margins=True)

## 4. Amounts

Amounts span many orders of magnitude, so they are compared on a log scale. Amounts are in
each transaction's own payment currency, so this is a rough comparison, not a converted one.

In [ ]:
tx.groupby("is_laundering")["amount_paid"].describe(percentiles=[0.1, 0.5, 0.9, 0.99])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.logspace(-6, 13, 80)
for label, colour in [(0, "#4C72B0"), (1, "#C44E52")]:
    ax.hist(tx.loc[tx["is_laundering"] == label, "amount_paid"], bins=bins, density=True,
            alpha=0.6, color=colour, label="laundering" if label else "normal")
ax.set_xscale("log")
ax.set_title("Amount paid by class (density, log scale)")
ax.legend()

## 5. Laundering patterns

In [ ]:
lines = load.load_pattern_lines()
attempts, current = [], None
for line in lines:
    if line.startswith("BEGIN LAUNDERING ATTEMPT"):
        current = {"typology": line.split(" - ", 1)[1].split(":")[0].strip(), "rows": []}
    elif line.startswith("END LAUNDERING ATTEMPT"):
        attempts.append(current)
        current = None
    elif line.strip():
        current["rows"].append(line)

attempt_sizes = pd.DataFrame(
    {"typology": [a["typology"] for a in attempts], "transactions": [len(a["rows"]) for a in attempts]}
)
attempt_sizes.groupby("typology")["transactions"].agg(attempts="size", total="sum", median="median", max="max")

In [ ]:
cols = load.TRANSACTION_COLUMNS

def show_attempt(typology, min_rows=5):
    """First attempt of a typology with at least `min_rows` transactions."""
    a = next(a for a in attempts if a["typology"] == typology and len(a["rows"]) >= min_rows)
    rows = pd.DataFrame([r.split(",") for r in a["rows"]], columns=cols)
    return rows[["timestamp", "from_bank", "from_account", "to_bank", "to_account",
                 "amount_paid", "payment_currency", "payment_format"]]

show_attempt("FAN-OUT")

In [ ]:
show_attempt("FAN-IN")

In [ ]:
show_attempt("CYCLE")

### Do the pattern blocks cover every laundering transaction?

Pattern lines use the same format as the transactions file, so each laundering row is
rebuilt as raw text and looked up in the pattern lines.

In [ ]:
raw = load.load_raw_transactions()
laundering_lines = raw.loc[raw["is_laundering"] == "1"].astype(str).agg(",".join, axis=1)
pattern_lines = {r for a in attempts for r in a["rows"]}
in_patterns = laundering_lines.isin(pattern_lines)
pd.Series(
    {
        "laundering transactions": len(laundering_lines),
        "found in a pattern block": int(in_patterns.sum()),
        "not in any pattern block": int((~in_patterns).sum()),
        "not in any pattern block (%)": round(100 * (~in_patterns).mean(), 2),
        "pattern-block lines": len(pattern_lines),
        "pattern-block lines not in transactions": len(pattern_lines - set(laundering_lines)),
    }
)

## 6. Account-level view: candidate mule labels

An account is identified by (bank ID, account number). The project scores accounts, so the
transaction-level flag has to be turned into an account-level label. Three candidates:

- **A. Any involvement:** sends or receives at least one laundering transaction.
- **B. Receive and forward:** both receives and sends at least one laundering transaction.
- **C. Receiving side:** receives at least one laundering transaction.

The denominator is every account that appears in at least one transaction.

In [ ]:
tx["from_id"] = tx["from_bank"] + ":" + tx["from_account"]
tx["to_id"] = tx["to_bank"] + ":" + tx["to_account"]
all_ids = pd.concat([tx["from_id"], tx["to_id"]]).drop_duplicates()

laundering = tx[tx["is_laundering"] == 1]
senders = set(laundering["from_id"])
receivers = set(laundering["to_id"])
self_laundering = int((laundering["from_id"] == laundering["to_id"]).sum())

candidates = {
    "A. any involvement": senders | receivers,
    "B. receive and forward": senders & receivers,
    "C. receiving side": receivers,
}
label_counts = pd.DataFrame(
    {
        "mule accounts": {k: len(v) for k, v in candidates.items()},
        "share of transacting accounts (%)": {
            k: 100 * len(v) / len(all_ids) for k, v in candidates.items()
        },
    }
)
print(f"transacting accounts: {len(all_ids):,}; laundering self-transfers: {self_laundering:,}")
label_counts

### How each candidate relates to the laundering roles

Accounts that only send laundering money are likely sources (where money enters), and
accounts that only receive it are likely destinations (where it ends up). Pass-through
accounts in the middle are closest to the usual idea of a mule.

In [ ]:
roles = pd.Series(
    {
        "send only": len(senders - receivers),
        "receive only": len(receivers - senders),
        "send and receive": len(senders & receivers),
    }
)
roles

In [ ]:
laundering_per_account = pd.concat([laundering["from_id"], laundering["to_id"]]).value_counts()
laundering_per_account.describe(percentiles=[0.5, 0.9, 0.99])